# Fine-tune Qwen (QLoRA) — trích xuất gia phả: Phả ký → entity + quan hệ (JSON)

> **Chỉ chạy trên Google Colab** — Runtime → Change runtime type → **GPU** (T4 chạy được nhưng chậm; **L4/A100** nên dùng), rồi chạy lần lượt từng cell.
> **Tốn compute units** (R3): chạy xong notebook tự **ngắt runtime** ở cell cuối.

**Dữ liệu** (sinh bởi `research/label_studio_pipeline/build_llm_sft_dataset.py`, thư mục `data/03_derived/llm_sft/v2/`, không có trong git):
`train.jsonl` (265 mẫu / 121 cây) · `val.jsonl` (22 / 13) · `test_silver.jsonl` (9 / 5 cây test) · `test_human.jsonl`.
Mỗi dòng: `{"messages": [system, user = đoạn Phả ký, assistant = JSON entities+relations], "meta": {...}}`.

**Nhãn là silver** (rule + sơ đồ phả hệ, Gemini web rà/bổ sung) — chưa người kiểm. Số đo trên `test_silver` = mức *khớp với nhãn Gemini*, **không phải** độ chính xác thật; số thật cần `check.csv` đã kiểm tay.

**Thí nghiệm:** cùng một model 4-bit, so **Qwen gốc (tắt LoRA)** vs **Qwen + LoRA** trên cùng tập test → micro P/R/F1 entity `(text, label)` và quan hệ `(type, head, tail)`, tỉ lệ JSON hợp lệ, tỉ lệ entity bịa (không có trong đoạn).

## Cell 0 — Cấu hình (sửa ở đây)

In [ ]:
# Model: Qwen2.5-7B-Instruct (Apache-2.0) chạy QLoRA vừa T4 16GB. Đổi sang model khác cùng họ nếu muốn
# (vd "Qwen/Qwen2.5-3B-Instruct" để chạy nhanh trên T4). Kiểm tra license trên model card trước khi dùng.
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# Nơi để 4 file jsonl. Mặc định đọc từ Google Drive; nếu không có sẽ hỏi upload ở Cell 2.
DRIVE_DATA_DIR = "/content/drive/MyDrive/family_tree/llm_sft/v2"
OUT_DIR = "/content/drive/MyDrive/family_tree/runs/qwen_qlora_v2"   # adapter + kết quả

MAX_LEN = 4096          # token tối đa / mẫu (prompt + đáp án). Mẫu dài hơn bị BỎ (không cắt JSON), có đếm.
EPOCHS = 3
LR = 2e-4
GRAD_ACCUM = 8          # batch hiệu dụng = 1 × GRAD_ACCUM
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
SEED = 42

EVAL_FILES = ["test_silver.jsonl"]   # thêm "val.jsonl" nếu đủ thời gian (sinh chậm trên T4)
MAX_NEW_TOKENS = 3072
DISCONNECT_AT_END = True             # ngắt runtime ở cell cuối để không tốn compute units

## Cell 1 — Cài thư viện + kiểm tra GPU

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "bitsandbytes", "accelerate"], check=True)
import torch, transformers, peft, bitsandbytes
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__,
      "| bitsandbytes", bitsandbytes.__version__)
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → GPU"
GPU = torch.cuda.get_device_name(0)
BF16 = torch.cuda.is_bf16_supported()   # T4 = False → dùng fp16
print("GPU:", GPU, "| bf16:", BF16, "| VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))

## Cell 2 — Lấy dữ liệu (Google Drive hoặc upload)

In [ ]:
import os, json, pathlib, shutil
DATA_DIR = pathlib.Path("/content/data"); DATA_DIR.mkdir(exist_ok=True)
NEED = ["train.jsonl", "val.jsonl", "test_silver.jsonl", "test_human.jsonl"]
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("Không mount được Drive:", exc)
src = pathlib.Path(DRIVE_DATA_DIR)
for name in NEED:
    if (src / name).exists():
        shutil.copy(src / name, DATA_DIR / name)
missing = [n for n in NEED if not (DATA_DIR / n).exists()]
if missing:
    print("Thiếu trên Drive:", missing, "→ chọn các file này để upload")
    from google.colab import files
    for name, blob in files.upload().items():
        (DATA_DIR / name).write_bytes(blob)
def read_jsonl(name):
    return [json.loads(l) for l in (DATA_DIR / name).read_text(encoding="utf-8").splitlines() if l.strip()]
data = {n: read_jsonl(n) for n in NEED if (DATA_DIR / n).exists()}
for n, rows in data.items():
    print(f"{n}: {len(rows)} mẫu, {len({r['meta']['tree_id'] for r in rows})} cây")
assert "train.jsonl" in data and "val.jsonl" in data, "Cần ít nhất train.jsonl và val.jsonl"
os.makedirs(OUT_DIR, exist_ok=True)

## Cell 3 — Load Qwen 4-bit + tokenizer, tạo mẫu train (chỉ tính loss trên phần đáp án)

In [ ]:
import random
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
random.seed(SEED); torch.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
COMPUTE_DTYPE = torch.bfloat16 if BF16 else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=COMPUTE_DTYPE, bnb_4bit_use_double_quant=True,
    ),
    torch_dtype=COMPUTE_DTYPE, device_map="auto",
)

def encode(example):
    """input_ids = chat template đầy đủ; labels = -100 ở phần system+user (chỉ học phần JSON đáp án)."""
    msgs = example["messages"]
    prompt_ids = tokenizer.apply_chat_template(msgs[:-1], add_generation_prompt=True, tokenize=True)
    full_ids = tokenizer.apply_chat_template(msgs, tokenize=True)
    assert full_ids[: len(prompt_ids)] == prompt_ids, "chat template không nối tiếp prompt → kiểm tra model"
    return {"input_ids": full_ids, "labels": [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]}

def build(rows, name):
    out, skipped = [], 0
    for r in rows:
        enc = encode(r)
        if len(enc["input_ids"]) > MAX_LEN:
            skipped += 1
            continue
        out.append(enc)
    lens = sorted(len(e["input_ids"]) for e in out)
    print(f"{name}: dùng {len(out)}, bỏ {skipped} mẫu > {MAX_LEN} token | token trung vị {lens[len(lens)//2]}, max {lens[-1]}")
    return out

train_ds = build(data["train.jsonl"], "train")
val_ds = build(data["val.jsonl"], "val")

## Cell 4 — Gắn LoRA + train

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
model.print_trainable_parameters()

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tokenizer.pad_token_id
    return {
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (width - len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (width - len(b["input_ids"])) for b in batch]),
        "labels": torch.tensor([b["labels"] + [-100] * (width - len(b["labels"])) for b in batch]),
    }

args = TrainingArguments(
    output_dir="/content/ckpt", num_train_epochs=EPOCHS, learning_rate=LR,
    per_device_train_batch_size=1, per_device_eval_batch_size=1, gradient_accumulation_steps=GRAD_ACCUM,
    lr_scheduler_type="cosine", warmup_ratio=0.05, logging_steps=5,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1, load_best_model_at_end=True,
    metric_for_best_model="eval_loss", greater_is_better=False,
    bf16=BF16, fp16=not BF16, optim="paged_adamw_8bit", gradient_checkpointing=True,
    report_to="none", seed=SEED, remove_unused_columns=False,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, data_collator=collate)
train_result = trainer.train()
print(train_result.metrics)
model.save_pretrained(f"{OUT_DIR}/adapter"); tokenizer.save_pretrained(f"{OUT_DIR}/adapter")
json.dump(trainer.state.log_history, open(f"{OUT_DIR}/log_history.json", "w"), ensure_ascii=False, indent=2)
print("Đã lưu adapter →", f"{OUT_DIR}/adapter")

## Cell 5 — Đánh giá: Qwen gốc vs Qwen + LoRA

Cùng model đã load: `with model.disable_adapter()` = Qwen gốc; bình thường = có LoRA. Sinh tham lam (không sampling).
So khớp **chính xác chuỗi**: entity `(text, label)`, quan hệ `(type, head, tail)` — chặt, nên tên đúng mà lệch một dấu cũng tính sai.

In [ ]:
import re, time, unicodedata
from collections import Counter

ANSWER_START = re.compile(r'\{\s*"entities"')
def parse_answer(text):
    """Lấy object {"entities"...} cuối cùng decode được (Qwen/Gemini đôi khi viết lại giữa chừng)."""
    dec = json.JSONDecoder()
    for m in reversed(list(ANSWER_START.finditer(text or ""))):
        try:
            obj, _ = dec.raw_decode(text, m.start())
            if isinstance(obj, dict):
                return obj
        except json.JSONDecodeError:
            continue
    return None

def norm(s):
    return unicodedata.normalize("NFC", str(s)).strip()

def as_sets(obj):
    ents = {(norm(e.get("text")), norm(e.get("label")).upper()) for e in obj.get("entities", []) if isinstance(e, dict)}
    rels = {(norm(r.get("type")).upper(), norm(r.get("head")), norm(r.get("tail"))) for r in obj.get("relations", []) if isinstance(r, dict)}
    return ents, rels

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    return {"P": round(p, 4), "R": round(r, 4), "F1": round(2 * p * r / (p + r), 4) if p + r else 0.0, "tp": tp, "fp": fp, "fn": fn}

@torch.no_grad()
def generate(example):
    ids = tokenizer.apply_chat_template(example["messages"][:-1], add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

def evaluate(rows, use_lora):
    c = Counter(); preds = []
    model.eval()
    for i, ex in enumerate(rows):
        t0 = time.time()
        if use_lora:
            raw = generate(ex)
        else:
            with model.disable_adapter():
                raw = generate(ex)
        gold_e, gold_r = as_sets(json.loads(ex["messages"][-1]["content"]))
        obj = parse_answer(raw)
        c["examples"] += 1
        if obj is None:
            c["invalid_json"] += 1; pred_e, pred_r = set(), set()
        else:
            pred_e, pred_r = as_sets(obj)
        chunk = norm(ex["messages"][1]["content"])
        c["pred_entities"] += len(pred_e)
        c["hallucinated_entities"] += sum(1 for t, _ in pred_e if t and t not in chunk)
        c["e_tp"] += len(pred_e & gold_e); c["e_fp"] += len(pred_e - gold_e); c["e_fn"] += len(gold_e - pred_e)
        c["r_tp"] += len(pred_r & gold_r); c["r_fp"] += len(pred_r - gold_r); c["r_fn"] += len(gold_r - pred_r)
        preds.append({"meta": ex["meta"], "raw": raw, "seconds": round(time.time() - t0, 1)})
        print(f"  {'LoRA' if use_lora else 'gốc'} {i+1}/{len(rows)} tree {ex['meta']['tree_id']} ({time.time()-t0:.0f}s)")
    metrics = {
        "entity": prf(c["e_tp"], c["e_fp"], c["e_fn"]),
        "relation": prf(c["r_tp"], c["r_fp"], c["r_fn"]),
        "json_valid_rate": round(1 - c["invalid_json"] / c["examples"], 4),
        "hallucinated_entity_rate": round(c["hallucinated_entities"] / c["pred_entities"], 4) if c["pred_entities"] else 0.0,
        "examples": c["examples"],
    }
    return metrics, preds

results = {"model": MODEL_ID, "gpu": GPU, "train_metrics": train_result.metrics, "eval": {}}
for name in EVAL_FILES:
    if name not in data:
        continue
    for use_lora in (False, True):
        tag = f"{name}:{'lora' if use_lora else 'base'}"
        print("==", tag)
        metrics, preds = evaluate(data[name], use_lora)
        results["eval"][tag] = metrics
        with open(f"{OUT_DIR}/pred_{name.replace('.jsonl', '')}_{'lora' if use_lora else 'base'}.jsonl", "w") as fh:
            for p in preds:
                fh.write(json.dumps(p, ensure_ascii=False) + "\n")
        print(json.dumps(metrics, ensure_ascii=False))
json.dump(results, open(f"{OUT_DIR}/results.json", "w"), ensure_ascii=False, indent=2)

## Cell 6 — Bảng so sánh

In [ ]:
print(f"{'tập:model':32} {'E-P':>6} {'E-R':>6} {'E-F1':>6} | {'R-P':>6} {'R-R':>6} {'R-F1':>6} | {'JSON ok':>7} {'bịa':>6}")
for tag, m in results["eval"].items():
    e, r = m["entity"], m["relation"]
    print(f"{tag:32} {e['P']:6.3f} {e['R']:6.3f} {e['F1']:6.3f} | {r['P']:6.3f} {r['R']:6.3f} {r['F1']:6.3f} | "
          f"{m['json_valid_rate']:7.2%} {m['hallucinated_entity_rate']:6.2%}")
print("\nNhắc lại: so với nhãn silver (Gemini rà), chưa phải độ chính xác thật. Test chỉ 5 cây → dao động lớn.")
print("Kết quả + adapter:", OUT_DIR)

## Cell 7 — Ngắt runtime (không tốn thêm compute units)

In [ ]:
# Kết quả đã nằm trên Drive (OUT_DIR). Đặt DISCONNECT_AT_END = False ở Cell 0 nếu muốn giữ runtime.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()